# Hybrid Retrieval Pipeline

A four-stage pipeline that progressively narrows a large corpus to a precise top-10:

```
  ┌─────────────────────────────────────────────────────────────────┐
  │  Phase 1 · Sparse (BM25) — domain-routed            → top-100  │
  │  Phase 2 · Dense  (FAISS + bge-large)               → top-100  │
  │  Phase 3 · RRF Fusion  (sparse ∪ dense)             → top-100  │
  │  Phase 4 · Cross-Encoder Reranker (bge-reranker)    → top-10   │
  └─────────────────────────────────────────────────────────────────┘
```

**Design principles**

| Stage | Goal | Key metric |
|-------|------|------------|
| Sparse | Maximum recall from keyword overlap | Recall@100 |
| Dense  | Semantic recall that BM25 misses    | Recall@100 |
| RRF    | Raise the recall ceiling via union  | Recall@100 |
| Rerank | Precision at the top                | NDCG@10    |

> **Domain routing** — the best sparse model is chosen *per domain*.  
> Because the routing decision must generalise to held-out queries you cannot
> see at inference time, **you manually set `DOMAIN_CONFIG`** in §1.3 based on
> the Recall@100 table printed by §1.2.

In [4]:
!pip install faiss-cpu

  Using cached faiss_cpu-1.13.2-cp310-abi3-macosx_14_0_arm64.whl.metadata (7.6 kB)
Using cached faiss_cpu-1.13.2-cp310-abi3-macosx_14_0_arm64.whl (3.5 MB)


---
## Imports & constants

In [28]:
import json
import math
import os
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR        = Path('../data')
SUBMISSIONS_DIR = Path('../submissions_heldout')
SUBMISSIONS_DIR.mkdir(exist_ok=True)

# ── Pipeline hyper-parameters ──────────────────────────────────────────
SPARSE_K   = 100   # top-k from sparse stage
DENSE_K    = 100   # top-k from dense stage
FUSION_K   = 100   # top-k kept after RRF fusion
FINAL_K    = 10    # top-k after cross-encoder reranking

RRF_C      = 60    # RRF constant  (standard = 60)
BGE_BATCH  = 32    # reduce if GPU OOM
CE_BATCH   = 16    # cross-encoder batch size

BGE_MODEL_ID = 'BAAI/bge-large-en-v1.5'
CE_MODEL_ID  = 'BAAI/bge-reranker-large'

print('Imports ready.')
print(f'SPARSE_K={SPARSE_K}  DENSE_K={DENSE_K}  FUSION_K={FUSION_K}  FINAL_K={FINAL_K}')

Imports ready.
SPARSE_K=100  DENSE_K=100  FUSION_K=100  FINAL_K=10


---
## Shared helpers

Utility functions shared across all four stages.

In [29]:
# ── Data loaders ──────────────────────────────────────────────────────
def load_queries(path): return pd.read_parquet(path)
def load_corpus(path):  return pd.read_parquet(path)
def load_qrels(path):
    with open(path) as f: return json.load(f)

# ── Text helpers ──────────────────────────────────────────────────────
def format_text(row):
    t = str(row.get('title','') or '').strip()
    a = str(row.get('abstract','') or '').strip()
    return (t + ' ' + a).strip() if (t and a) else (t or a)

def get_ta(row):
    return str(row.get('ta','') or '').strip()

# ── Submission builder ────────────────────────────────────────────────
def make_submission(score_matrix, q_ids, c_ids, top_k=100):
    return {q_ids[i]: [c_ids[j] for j in np.argsort(-score_matrix[i])[:top_k]]
            for i in range(len(q_ids))}

# ── Disk cache helpers ────────────────────────────────────────────────
def _is_cached(name):
    return ((SUBMISSIONS_DIR / f'submission_{name}.json').exists() and
            (SUBMISSIONS_DIR / f'scores_{name}.npy').exists())

def _load_cache(name):
    with open(SUBMISSIONS_DIR / f'submission_{name}.json') as _f:
        sub = json.load(_f)
    scores = np.load(str(SUBMISSIONS_DIR / f'scores_{name}.npy'))
    return sub, scores

def _save_cache(name, sub, scores):
    with open(SUBMISSIONS_DIR / f'submission_{name}.json', 'w') as _f:
        json.dump(sub, _f)
    np.save(str(SUBMISSIONS_DIR / f'scores_{name}.npy'), scores)

# ── Metric helpers ────────────────────────────────────────────────────
def recall_at_k(ranked, relevant, k):
    if not relevant: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / len(relevant)

def precision_at_k(ranked, relevant, k):
    if k == 0: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / k

def mrr_at_k(ranked, relevant, k):
    for rank, doc in enumerate(ranked[:k], 1):
        if doc in relevant: return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked, relevant, k):
    dcg  = sum(1/math.log2(r+1) for r,d in enumerate(ranked[:k],1) if d in relevant)
    idcg = sum(1/math.log2(r+1) for r in range(1, min(len(relevant),k)+1))
    return dcg/idcg if idcg else 0.0

def average_precision(ranked, relevant):
    if not relevant: return 0.0
    hits = score = 0
    for rank, doc in enumerate(ranked, 1):
        if doc in relevant:
            hits += 1; score += hits/rank
    return score / len(relevant)

def evaluate(submission, qrels, ks=None, query_domains=None, verbose=True):
    if ks is None: ks = [10, 50, 100]
    per_query = {}
    for qid, rel_list in qrels.items():
        relevant = set(rel_list)
        ranked   = submission.get(qid, [])
        q = {}
        for k in ks:
            q[f'Recall@{k}']    = recall_at_k(ranked, relevant, k)
            q[f'Precision@{k}'] = precision_at_k(ranked, relevant, k)
            q[f'MRR@{k}']       = mrr_at_k(ranked, relevant, k)
            q[f'NDCG@{k}']      = ndcg_at_k(ranked, relevant, k)
        q['AP'] = average_precision(ranked, relevant)
        per_query[qid] = q

    metric_keys = list(next(iter(per_query.values())).keys()) if per_query else []
    overall = {key: float(np.mean([per_query[q][key] for q in per_query])) for key in metric_keys}
    overall['MAP'] = overall.pop('AP', 0.0)
    overall['num_queries'] = len(per_query)
    result = {'overall': overall, 'per_query': per_query}

    if query_domains:
        per_domain = {}
        for domain in sorted(set(query_domains.values())):
            dqids = [q for q in per_query if query_domains.get(q) == domain]
            if not dqids: continue
            dm = {k: float(np.mean([per_query[q][k] for q in dqids])) for k in metric_keys}
            dm['MAP'] = dm.pop('AP', 0.0)
            dm['num_queries'] = len(dqids)
            per_domain[domain] = dm
        result['per_domain'] = per_domain

    if verbose:
        _print_results(result, ks)
    return result

def _print_results(results, ks):
    o = results['overall']
    print('\n' + '='*68 + '\nOVERALL RESULTS\n' + '='*68)
    for label, keys in [('Recall',    [f'Recall@{k}'    for k in ks]),
                        ('Precision', [f'Precision@{k}' for k in ks]),
                        ('MRR',       [f'MRR@{k}'       for k in ks]),
                        ('NDCG',      [f'NDCG@{k}'      for k in ks])]:
        print(f'{label:<14}' + ''.join(f'  @{k:>3}: {o.get(f,0):.4f}' for k,f in zip(ks,keys)))
    print(f"{'MAP':<14}  {o.get('MAP',0):.4f}")
    print(f"{'Queries':<14}  {int(o.get('num_queries',0))}")
    if 'per_domain' in results:
        k = ks[0]
        print(f'\n{"-"*68}\nPER-DOMAIN  (NDCG@{k})\n{"-"*68}')
        print(f"  {'Domain':<28} {'NDCG@'+str(k):<10} {'Recall@'+str(k):<12} MAP")
        for domain, dm in sorted(results['per_domain'].items()):
            print(f"  {domain:<28} {dm.get(f'NDCG@{k}',0):.4f}     "
                  f"{dm.get(f'Recall@{k}',0):.4f}       {dm.get('MAP',0):.4f}")
    print()

def compare_submissions(submissions, qrels, ks=None, query_domains=None):
    if ks is None: ks = [10, 50, 100]
    names   = list(submissions.keys())
    results = {n: evaluate(s, qrels, ks=ks, query_domains=query_domains, verbose=False)
               for n, s in submissions.items()}
    overall = {n: results[n]['overall'] for n in names}
    metric_keys = [k for k in next(iter(overall.values())) if k != 'num_queries']

    col_w  = max(len(n) for n in names) + 2
    header = f"{'Metric':<20}" + ''.join(f'{n:>{col_w}}' for n in names) + '  Best'
    if len(names) == 2: header += f"  {'Delta':>8}"
    print('\n' + '='*len(header) + '\nSUBMISSION COMPARISON\n' + '='*len(header))
    print(header); print('-'*len(header))
    for metric in metric_keys:
        vals = {n: round(overall[n].get(metric,0.0),4) for n in names}
        best = max(vals, key=vals.get)
        line = f'{metric:<20}' + ''.join(f'{vals[n]:>{col_w}.4f}' for n in names) + f'  {best}'
        if len(names) == 2:
            a,b = names
            line += f'  {vals[b]-vals[a]:>+8.4f}'
        print(line)
    print()
    return results

print('Helpers ready.')

Helpers ready.


---
## Data loading

In [23]:
queries = load_queries(DATA_DIR / 'queries.parquet')
corpus  = load_corpus(DATA_DIR  / 'corpus.parquet')
qrels   = load_qrels(DATA_DIR   / 'qrels_1.json')

query_ids     = queries['doc_id'].tolist()
corpus_ids    = corpus['doc_id'].tolist()
query_domains = dict(zip(queries['doc_id'], queries['domain']))

corpus_texts = [format_text(row) for _, row in tqdm(corpus.iterrows(),  total=len(corpus),  desc='corpus TA')]
query_texts  = [format_text(row) for _, row in tqdm(queries.iterrows(), total=len(queries), desc='query TA')]

print(f'\nQueries : {len(queries)}  |  Corpus : {len(corpus)}  |  Qrels : {len(qrels)}')

query TA: 100%|██████████| 100/100 [00:00<00:00, 14301.85it/s]


Queries : 100  |  Corpus : 20000  |  Qrels : 100


---
# Phase 1 — Sparse Retrieval (load from bm25.ipynb)

The routed sparse submission is produced by `bm25.ipynb` and written to
`submissions/submission_sparse_routed_top100.json`. Load it here directly —
no sparse models need to be re-fitted in this notebook.

In [24]:
_sparse_route_path = SUBMISSIONS_DIR / f'submission_sparse_routed_top{SPARSE_K}.json'

if not _sparse_route_path.exists():
    raise FileNotFoundError(
        f'{_sparse_route_path} not found.\n'
        'Run bm25.ipynb first to generate the routed sparse submission.'
    )

with open(_sparse_route_path) as _f:
    sub_sparse_routed = json.load(_f)

print(f'Sparse routed (top-{SPARSE_K}): loaded from bm25.ipynb output. ✓')
print(f'  Queries covered: {len(sub_sparse_routed)}')

Sparse routed (top-100): loaded from bm25.ipynb output. ✓
  Queries covered: 100


---
# Phase 2 — Dense Retrieval (FAISS + bge-large)

A bi-encoder retrieves semantically similar documents that BM25 would miss due to
vocabulary mismatch — e.g. "myocardial infarction" vs "heart attack".

**Model:** [`BAAI/bge-large-en-v1.5`](https://huggingface.co/BAAI/bge-large-en-v1.5)  
A 335 M-parameter model that produces 1024-dimensional L2-normalised embeddings.
It is the strongest open English bi-encoder at this size class.

**Index:** A FAISS `IndexFlatIP` (exact inner-product search on normalised vectors
= cosine similarity).  Switch to `IndexIVFFlat` / `IndexHNSW` if the corpus is too
large to fit the flat index in memory.

**Query instruction:** bge-large is trained with an instruction prefix for retrieval
queries: `"Represent this sentence for searching relevant passages: "`.  Corpus
documents are embedded *without* the prefix.

## §2.1 — Compute / load bge-large embeddings

In [26]:
from sentence_transformers import SentenceTransformer

_bge_heldout_query_cache  = SUBMISSIONS_DIR / 'bge_large_heldout_query_emb.npy'

BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

_bge = SentenceTransformer(BGE_MODEL_ID)

print('Embedding queries (with instruction prefix) ...')
_query_texts_prefixed = [BGE_QUERY_PREFIX + t for t in query_texts]
bge_query_emb = _bge.encode(
    _query_texts_prefixed,
    batch_size=BGE_BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype(np.float32)

np.save(str(_bge_heldout_query_cache),  bge_query_emb)
print(f'Saved — query: {bge_query_emb.shape}  ✓')

Embedding queries (with instruction prefix) ...


Batches: 100%|██████████| 4/4 [00:41<00:00, 10.42s/it]

Saved — query: (100, 1024)  ✓


In [30]:
from sentence_transformers import SentenceTransformer

_bge_corpus_cache = SUBMISSIONS_DIR / 'bge_large_corpus_emb.npy'
_bge_query_cache  = SUBMISSIONS_DIR / 'bge_large_query_emb.npy'

# bge-large uses a task instruction prefix on the query side only
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

if _bge_corpus_cache.exists() and _bge_query_cache.exists():
    print('Loading bge-large embeddings from cache ...')
    bge_corpus_emb = np.load(str(_bge_corpus_cache))
    bge_query_emb  = np.load(str(_bge_query_cache))
    print(f'  corpus: {bge_corpus_emb.shape}  query: {bge_query_emb.shape}  ✓')
else:
    print(f'Loading {BGE_MODEL_ID} ...')
    _bge = SentenceTransformer(BGE_MODEL_ID)

    print('Embedding corpus (no prefix) ...')
    bge_corpus_emb = _bge.encode(
        corpus_texts,
        batch_size=BGE_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    print('Embedding queries (with instruction prefix) ...')
    _query_texts_prefixed = [BGE_QUERY_PREFIX + t for t in query_texts]
    bge_query_emb = _bge.encode(
        _query_texts_prefixed,
        batch_size=BGE_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    np.save(str(_bge_corpus_cache), bge_corpus_emb)
    np.save(str(_bge_query_cache),  bge_query_emb)
    print(f'Saved — corpus: {bge_corpus_emb.shape}  query: {bge_query_emb.shape}  ✓')

Loading bge-large embeddings from cache ...
  corpus: (20000, 1024)  query: (100, 1024)  ✓


## §2.2 — Build FAISS index and retrieve top-100

In [ ]:
_dense_cache = SUBMISSIONS_DIR / f'submission_heldout_bge_dense_top{DENSE_K}.json'

if _dense_cache.exists():
    with open(_dense_cache) as _f:
        sub_dense = json.load(_f)
    print(f'Dense retrieval (top-{DENSE_K}): loaded from cache. ✓')
else:
    dim = bge_corpus_emb.shape[1]
    print(f'Building FAISS IndexFlatIP  dim={dim}  corpus={len(corpus_ids):,} ...')

    _index = faiss.IndexFlatIP(dim)
    _index.add(bge_corpus_emb)
    print(f'  Index contains {_index.ntotal:,} vectors.')

    print(f'Searching top-{DENSE_K} for {len(query_ids)} queries ...')
    _D, _I = _index.search(bge_query_emb, DENSE_K)  # (Q, DENSE_K)

    _cids = np.array(corpus_ids)
    sub_dense = {
        query_ids[i]: _cids[_I[i]].tolist()
        for i in range(len(query_ids))
    }

    with open(_dense_cache, 'w') as _f:
        json.dump(sub_dense, _f)
    print(f'Dense retrieval (top-{DENSE_K}): saved. ✓')

print()
evaluate(sub_dense, qrels, ks=[10, 50, 100], query_domains=query_domains)

Building FAISS IndexFlatIP  dim=1024  corpus=20,000 ...
  Index contains 20,000 vectors.
Searching top-100 for 100 queries ...
Dense retrieval (top-100): saved. ✓


OVERALL RESULTS
Recall          @ 10: 0.0000  @ 50: 0.0000  @100: 0.0000
Precision       @ 10: 0.0000  @ 50: 0.0000  @100: 0.0000
MRR             @ 10: 0.0000  @ 50: 0.0000  @100: 0.0000
NDCG            @ 10: 0.0000  @ 50: 0.0000  @100: 0.0000
MAP             0.0000
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (NDCG@10)
--------------------------------------------------------------------
  Domain                       NDCG@10    Recall@10    MAP



{'overall': {'Recall@10': 0.0,
  'Precision@10': 0.0,
  'MRR@10': 0.0,
  'NDCG@10': 0.0,
  'Recall@50': 0.0,
  'Precision@50': 0.0,
  'MRR@50': 0.0,
  'NDCG@50': 0.0,
  'Recall@100': 0.0,
  'Precision@100': 0.0,
  'MRR@100': 0.0,
  'NDCG@100': 0.0,
  'MAP': 0.0,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.0,
   'Precision@10': 0.0,
   'MRR@10': 0.0,
   'NDCG@10': 0.0,
   'Recall@50': 0.0,
   'Precision@50': 0.0,
   'MRR@50': 0.0,
   'NDCG@50': 0.0,
   'Recall@100': 0.0,
   'Precision@100': 0.0,
   'MRR@100': 0.0,
   'NDCG@100': 0.0,
   'AP': 0.0},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.0,
   'Precision@10': 0.0,
   'MRR@10': 0.0,
   'NDCG@10': 0.0,
   'Recall@50': 0.0,
   'Precision@50': 0.0,
   'MRR@50': 0.0,
   'NDCG@50': 0.0,
   'Recall@100': 0.0,
   'Precision@100': 0.0,
   'MRR@100': 0.0,
   'NDCG@100': 0.0,
   'AP': 0.0},
  '021ff5083be85ee7bd0f828457dc46c90213b7e7': {'Recall@10': 0.0,
   'Precision@10':

## §2.3 — Sparse vs. Dense: complementarity check

How much does each stage contribute that the other misses?
High per-stage overlap means fusion will not help much; low overlap means RRF has
room to raise recall.

In [11]:
compare_submissions(
    {f'Sparse routed (top-{SPARSE_K})': sub_sparse_routed,
     f'Dense bge-large (top-{DENSE_K})': sub_dense},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)

# ── Overlap analysis ──────────────────────────────────────────────────────
_overlaps = []
for qid in query_ids:
    sp = set(sub_sparse_routed.get(qid, []))
    de = set(sub_dense.get(qid, []))
    if sp or de:
        _overlaps.append(len(sp & de) / len(sp | de))

print(f'Mean Jaccard overlap (sparse ∩ dense) / (sparse ∪ dense)  =  {np.mean(_overlaps):.4f}')
print(f'Mean union size per query  =  '
      f'{np.mean([len(set(sub_sparse_routed.get(q,[])) | set(sub_dense.get(q,[]))) for q in query_ids]):.1f} docs')


SUBMISSION COMPARISON
Metric                  Sparse routed (top-100)  Dense bge-large (top-100)  Best     Delta
------------------------------------------------------------------------------------------
Recall@10                                0.5241                     0.5212  Sparse routed (top-100)   -0.0029
Precision@10                             0.2750                     0.2700  Sparse routed (top-100)   -0.0050
MRR@10                                   0.6631                     0.7250  Dense bge-large (top-100)   +0.0619
NDCG@10                                  0.5433                     0.5580  Dense bge-large (top-100)   +0.0147
Recall@50                                0.7681                     0.7261  Sparse routed (top-100)   -0.0420
Precision@50                             0.1036                     0.1000  Sparse routed (top-100)   -0.0036
MRR@50                                   0.6683                     0.7262  Dense bge-large (top-100)   +0.0579
NDCG@50            

---
# Phase 3 — RRF Fusion

**Reciprocal Rank Fusion (RRF)** merges two ranked lists without needing to
normalise or calibrate raw scores.  For each document $d$ that appears in any
of the $L$ ranked lists:

$$\text{RRF}(d) = \sum_{l=1}^{L} \frac{1}{c + r_l(d)}$$

where $r_l(d)$ is the rank of $d$ in list $l$ (1-indexed), and $c$ is a constant
(default 60) that dampens the influence of very high ranks.

If a document is absent from a list its contribution from that list is zero
(equivalent to $r_l(d) = \infty$).

**Why RRF here?**
- Score-space fusion (e.g. $\alpha \cdot s_{\text{sparse}} + (1-\alpha) \cdot s_{\text{dense}}$)
  requires calibrated scores; BM25 and cosine similarity live on incompatible scales.
- RRF is robust to rank-score mis-calibration and consistently matches or beats
  score-fusion in retrieval benchmarks (Cormack et al., 2009).
- The union of the two top-100 lists gives a pool of up to 200 unique documents;
  RRF re-ranks this pool and we keep the top `FUSION_K`.

In [12]:
_rrf_cache = SUBMISSIONS_DIR / f'submission_rrf_top{FUSION_K}.json'

if _rrf_cache.exists():
    with open(_rrf_cache) as _f:
        sub_rrf = json.load(_f)
    print(f'RRF fusion (top-{FUSION_K}): loaded from cache. ✓')
else:
    print(f'Running RRF fusion  c={RRF_C}  FUSION_K={FUSION_K} ...')
    sub_rrf = {}

    for qid in tqdm(query_ids, desc='RRF fusion'):
        sparse_list = sub_sparse_routed.get(qid, [])
        dense_list  = sub_dense.get(qid, [])

        rrf_scores: dict[str, float] = {}
        for rank, doc_id in enumerate(sparse_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_C + rank)
        for rank, doc_id in enumerate(dense_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_C + rank)

        sub_rrf[qid] = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:FUSION_K]

    with open(_rrf_cache, 'w') as _f:
        json.dump(sub_rrf, _f)
    print(f'RRF fusion (top-{FUSION_K}): saved. ✓')

print()
evaluate(sub_rrf, qrels, ks=[10, 50, 100], query_domains=query_domains)

Running RRF fusion  c=60  FUSION_K=100 ...


RRF fusion: 100%|██████████| 100/100 [00:00<00:00, 11263.51it/s]

RRF fusion (top-100): saved. ✓


OVERALL RESULTS
Recall          @ 10: 0.5527  @ 50: 0.7997  @100: 0.8661
Precision       @ 10: 0.2810  @ 50: 0.1078  @100: 0.0640
MRR             @ 10: 0.7339  @ 50: 0.7349  @100: 0.7351
NDCG            @ 10: 0.5798  @ 50: 0.6367  @100: 0.6576
MAP             0.4789
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (NDCG@10)
--------------------------------------------------------------------
  Domain                       NDCG@10    Recall@10    MAP
  Art                          0.5585     0.5000       0.3750
  Biology                      0.5454     0.5311       0.4308
  Business                     0.7197     0.5667       0.6107
  Chemistry                    0.6692     0.4786       0.5401
  Computer Science             0.4462     0.3236       0.3885
  Economics                    0.8669     0.4706       0.7276
  Engineering                  0.5215     0.5000       0.3561
  Environmental Science   

{'overall': {'Recall@10': 0.552667939721519,
  'Precision@10': 0.281,
  'MRR@10': 0.7339285714285714,
  'NDCG@10': 0.579818376276357,
  'Recall@50': 0.7997015650983107,
  'Precision@50': 0.10779999999999999,
  'MRR@50': 0.7349301587301588,
  'NDCG@50': 0.6367135638355151,
  'Recall@100': 0.8661481906382931,
  'Precision@100': 0.064,
  'MRR@100': 0.73510559732665,
  'NDCG@100': 0.6576049771503264,
  'MAP': 0.4789169510958573,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
   'Precision@10': 0.2,
   'MRR@10': 1.0,
   'NDCG@10': 0.5585075862632192,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.5585075862632192,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.5585075862632192,
   'AP': 0.375},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.5714285714285714,
   'Precision@10': 0.8,
   'MRR@10': 1.0,
   'NDCG@10': 0.8521705090845474,
   'Recall@50': 0.857

In [13]:
# ── Stage comparison: sparse → dense → RRF ───────────────────────────────
compare_submissions(
    {
        f'Sparse routed (top-{SPARSE_K})':    sub_sparse_routed,
        f'Dense bge-large (top-{DENSE_K})':   sub_dense,
        f'RRF fusion (top-{FUSION_K})':        sub_rrf,
    },
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)

# Recall@100 ceiling going into the reranker
_r100 = float(np.mean([
    recall_at_k(sub_rrf.get(qid, []), set(qrels[qid]), 100)
    for qid in qrels if qrels[qid]
]))
print(f'Recall@100 ceiling for reranker input  =  {_r100:.4f}')
print(f'(This is the upper bound on Recall@100 achievable by the cross-encoder.)')


SUBMISSION COMPARISON
Metric                  Sparse routed (top-100)  Dense bge-large (top-100)       RRF fusion (top-100)  Best
-----------------------------------------------------------------------------------------------------------
Recall@10                                0.5241                     0.5212                     0.5527  RRF fusion (top-100)
Precision@10                             0.2750                     0.2700                     0.2810  RRF fusion (top-100)
MRR@10                                   0.6631                     0.7250                     0.7339  RRF fusion (top-100)
NDCG@10                                  0.5433                     0.5580                     0.5798  RRF fusion (top-100)
Recall@50                                0.7681                     0.7261                     0.7997  RRF fusion (top-100)
Precision@50                             0.1036                     0.1000                     0.1078  RRF fusion (top-100)
MRR@50           

---
# Phase 4 — Cross-Encoder Reranking

A cross-encoder processes each (query, document) pair jointly — unlike bi-encoders
it can model fine-grained query-document interactions.  This makes it significantly
more accurate but also much slower: $O(N)$ forward passes per query.  We therefore
apply it only to the `FUSION_K = 100` candidates from Phase 3.

**Model:** [`BAAI/bge-reranker-large`](https://huggingface.co/BAAI/bge-reranker-large)  
A fine-tuned cross-encoder (435 M parameters) that returns a relevance logit for
each (query, passage) pair.  It is the strongest open cross-encoder in its size
class and pairs naturally with bge-large bi-encoder.

**Input format:** `[query text, document text]` — the model tokenises both together.
We truncate each document to `max_length=512` tokens.

**Output:** top-`FINAL_K` (= 10) documents per query, sorted by relevance logit.

## §4.1 — Load cross-encoder

In [14]:
from sentence_transformers import CrossEncoder

print(f'Loading cross-encoder: {CE_MODEL_ID} ...')
_ce_model = CrossEncoder(CE_MODEL_ID, max_length=512)
print('Cross-encoder ready. ✓')

Loading cross-encoder: BAAI/bge-reranker-large ...
Cross-encoder ready. ✓


## §4.2 — Build document text lookup

The cross-encoder needs the full document text for each candidate.  We build an
in-memory map `corpus_id → text` using title + abstract (extending to full text
is straightforward but bumps latency proportionally to text length).

In [15]:
# Map corpus_id → title+abstract text (fast lookup during reranking)
_cid_to_text = {
    row['doc_id']: format_text(row)
    for _, row in tqdm(corpus.iterrows(), total=len(corpus), desc='Building doc lookup')
}

# Map query_id → query text
_qid_to_text = {
    row['doc_id']: format_text(row)
    for _, row in queries.iterrows()
}

print(f'Lookup tables built: {len(_cid_to_text):,} docs  |  {len(_qid_to_text):,} queries.')

Building doc lookup: 100%|██████████| 20000/20000 [00:08<00:00, 2464.42it/s]


Lookup tables built: 20,000 docs  |  100 queries.


## §4.3 — Rerank RRF candidates → top-10

In [16]:
_ce_cache = SUBMISSIONS_DIR / f'submission_ce_reranked_top{FINAL_K}.json'

if _ce_cache.exists():
    with open(_ce_cache) as _f:
        sub_reranked = json.load(_f)
    print(f'Cross-encoder reranked (top-{FINAL_K}): loaded from cache. ✓')
else:
    print(f'Cross-encoder reranking {FUSION_K} candidates → top-{FINAL_K} ...')
    sub_reranked = {}

    for qid in tqdm(query_ids, desc=f'CE reranking (top-{FINAL_K})'):
        candidates = sub_rrf.get(qid, [])
        if not candidates:
            sub_reranked[qid] = []
            continue

        q_text  = _qid_to_text.get(qid, '')
        pairs   = [[q_text, _cid_to_text.get(cid, '')] for cid in candidates]
        # predict() returns a score per pair; higher = more relevant
        logits  = _ce_model.predict(pairs, batch_size=CE_BATCH, show_progress_bar=False)
        top_idx = np.argsort(-logits)[:FINAL_K]
        sub_reranked[qid] = [candidates[i] for i in top_idx]

    with open(_ce_cache, 'w') as _f:
        json.dump(sub_reranked, _f)
    print(f'Cross-encoder reranked (top-{FINAL_K}): saved. ✓')

print()
evaluate(sub_reranked, qrels, ks=[1, 5, 10], query_domains=query_domains)

Cross-encoder reranking 100 candidates → top-10 ...


CE reranking (top-10): 100%|██████████| 100/100 [52:53<00:00, 31.74s/it]

Cross-encoder reranked (top-10): saved. ✓


OVERALL RESULTS
Recall          @  1: 0.0886  @  5: 0.2381  @ 10: 0.3604
Precision       @  1: 0.3500  @  5: 0.2300  @ 10: 0.1850
MRR             @  1: 0.3500  @  5: 0.4507  @ 10: 0.4772
NDCG            @  1: 0.3500  @  5: 0.3194  @ 10: 0.3513
MAP             0.2011
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (NDCG@1)
--------------------------------------------------------------------
  Domain                       NDCG@1     Recall@1     MAP
  Art                          1.0000     0.2500       0.2500
  Biology                      0.1429     0.0519       0.1375
  Business                     0.5000     0.0333       0.1613
  Chemistry                    0.5000     0.0562       0.1591
  Computer Science             0.4167     0.0507       0.2018
  Economics                    1.0000     0.0588       0.2941
  Engineering                  0.0000     0.0000       0.0250
  Environmental 

{'overall': {'Recall@1': 0.08863154182213559,
  'Precision@1': 0.35,
  'MRR@1': 0.35,
  'NDCG@1': 0.35,
  'Recall@5': 0.2380576820564833,
  'Precision@5': 0.23,
  'MRR@5': 0.4506666666666667,
  'NDCG@5': 0.31935720505421267,
  'Recall@10': 0.36039137186635906,
  'Precision@10': 0.18500000000000003,
  'MRR@10': 0.47722619047619047,
  'NDCG@10': 0.35126074404550905,
  'MAP': 0.20107094954807475,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@1': 0.25,
   'Precision@1': 1.0,
   'MRR@1': 1.0,
   'NDCG@1': 1.0,
   'Recall@5': 0.25,
   'Precision@5': 0.2,
   'MRR@5': 1.0,
   'NDCG@5': 0.3903800499921017,
   'Recall@10': 0.25,
   'Precision@10': 0.1,
   'MRR@10': 1.0,
   'NDCG@10': 0.3903800499921017,
   'AP': 0.25},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@1': 0.07142857142857142,
   'Precision@1': 1.0,
   'MRR@1': 1.0,
   'NDCG@1': 1.0,
   'Recall@5': 0.21428571428571427,
   'Precision@5': 0.6,
   'MRR@5': 1.0,
   'NDCG@5': 0.7227265

---
# Final evaluation — end-to-end pipeline

Full stage-by-stage comparison showing how each phase contributes.

In [17]:
compare_submissions(
    {
        f'1 · Sparse routed  (top-{SPARSE_K})':  sub_sparse_routed,
        f'2 · Dense bge-large (top-{DENSE_K})':  sub_dense,
        f'3 · RRF fusion      (top-{FUSION_K})':  sub_rrf,
        f'4 · CE reranked     (top-{FINAL_K})':   sub_reranked,
    },
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)


SUBMISSION COMPARISON
Metric                 1 · Sparse routed  (top-100)  2 · Dense bge-large (top-100)  3 · RRF fusion      (top-100)   4 · CE reranked     (top-10)  Best
------------------------------------------------------------------------------------------------------------------------------------------------------
Recall@10                                    0.5241                         0.5212                         0.5527                         0.3604  3 · RRF fusion      (top-100)
Precision@10                                 0.2750                         0.2700                         0.2810                         0.1850  3 · RRF fusion      (top-100)
MRR@10                                       0.6631                         0.7250                         0.7339                         0.4772  3 · RRF fusion      (top-100)
NDCG@10                                      0.5433                         0.5580                         0.5798                         0.3513  3

{'1 · Sparse routed  (top-100)': {'overall': {'Recall@10': 0.5240845250013657,
   'Precision@10': 0.275,
   'MRR@10': 0.6631468253968253,
   'NDCG@10': 0.5432998196352825,
   'Recall@50': 0.7681231661911871,
   'Precision@50': 0.10360000000000001,
   'MRR@50': 0.6682644884703709,
   'NDCG@50': 0.5996678754409109,
   'Recall@100': 0.8522425824401685,
   'Precision@100': 0.0614,
   'MRR@100': 0.6682644884703709,
   'NDCG@100': 0.6230909435007108,
   'MAP': 0.4450587923291623,
   'num_queries': 100},
  'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
    'Precision@10': 0.2,
    'MRR@10': 1.0,
    'NDCG@10': 0.5078961547485289,
    'Recall@50': 0.5,
    'Precision@50': 0.04,
    'MRR@50': 1.0,
    'NDCG@50': 0.5078961547485289,
    'Recall@100': 0.5,
    'Precision@100': 0.02,
    'MRR@100': 1.0,
    'NDCG@100': 0.5078961547485289,
    'AP': 0.3055555555555556},
   '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.5,
    'Precision@10': 0.7,
    'MRR@

In [18]:
# ── Per-domain breakdown of the final reranked submission ────────────────
evaluate(sub_reranked, qrels, ks=[1, 5, 10], query_domains=query_domains)


OVERALL RESULTS
Recall          @  1: 0.0886  @  5: 0.2381  @ 10: 0.3604
Precision       @  1: 0.3500  @  5: 0.2300  @ 10: 0.1850
MRR             @  1: 0.3500  @  5: 0.4507  @ 10: 0.4772
NDCG            @  1: 0.3500  @  5: 0.3194  @ 10: 0.3513
MAP             0.2011
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (NDCG@1)
--------------------------------------------------------------------
  Domain                       NDCG@1     Recall@1     MAP
  Art                          1.0000     0.2500       0.2500
  Biology                      0.1429     0.0519       0.1375
  Business                     0.5000     0.0333       0.1613
  Chemistry                    0.5000     0.0562       0.1591
  Computer Science             0.4167     0.0507       0.2018
  Economics                    1.0000     0.0588       0.2941
  Engineering                  0.0000     0.0000       0.0250
  Environmental Science        0.6667     0.0905       0.17

{'overall': {'Recall@1': 0.08863154182213559,
  'Precision@1': 0.35,
  'MRR@1': 0.35,
  'NDCG@1': 0.35,
  'Recall@5': 0.2380576820564833,
  'Precision@5': 0.23,
  'MRR@5': 0.4506666666666667,
  'NDCG@5': 0.31935720505421267,
  'Recall@10': 0.36039137186635906,
  'Precision@10': 0.18500000000000003,
  'MRR@10': 0.47722619047619047,
  'NDCG@10': 0.35126074404550905,
  'MAP': 0.20107094954807475,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@1': 0.25,
   'Precision@1': 1.0,
   'MRR@1': 1.0,
   'NDCG@1': 1.0,
   'Recall@5': 0.25,
   'Precision@5': 0.2,
   'MRR@5': 1.0,
   'NDCG@5': 0.3903800499921017,
   'Recall@10': 0.25,
   'Precision@10': 0.1,
   'MRR@10': 1.0,
   'NDCG@10': 0.3903800499921017,
   'AP': 0.25},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@1': 0.07142857142857142,
   'Precision@1': 1.0,
   'MRR@1': 1.0,
   'NDCG@1': 1.0,
   'Recall@5': 0.21428571428571427,
   'Precision@5': 0.6,
   'MRR@5': 1.0,
   'NDCG@5': 0.7227265

---
## Inference on held-out queries

Run the full four-stage pipeline on a new query set (no qrels required).

Steps:
1. Load the held-out query parquet.
2. Set `DOMAIN_CONFIG` domain assignments for the new domains (re-use §1.3 config).
3. Run this cell — it will produce `submission_final_top10.json`.

In [20]:
def run_pipeline(
    new_queries_path: str,
    sparse_path: str = str(SUBMISSIONS_DIR / f'submission_sparse_routed_top{SPARSE_K}.json'),
    output_path: str = str(SUBMISSIONS_DIR / 'submission_final_top10.json'),
):
    """End-to-end pipeline for held-out queries (no qrels needed).

    Args:
        new_queries_path : path to held-out queries parquet
        sparse_path      : path to routed sparse JSON written by bm25.ipynb
        output_path      : where to save the final top-10 submission JSON
    """
    from sentence_transformers import SentenceTransformer, CrossEncoder

    new_queries   = load_queries(new_queries_path)
    new_q_ids     = new_queries['doc_id'].tolist()
    new_q_texts   = [format_text(row) for _, row in new_queries.iterrows()]

    print(f'Loaded {len(new_q_ids)} held-out queries.')

    # Stage 1: load pre-computed sparse from bm25.ipynb
    print('\nStage 1 — Loading sparse routed submission ...')
    if not Path(sparse_path).exists():
        raise FileNotFoundError(
            f'{sparse_path} not found.\n'
            'Run bm25.ipynb on the held-out queries first.'
        )
    with open(sparse_path) as _f:
        new_sparse = json.load(_f)
    print(f'  Loaded {len(new_sparse)} queries. ✓')

    # Stage 2: dense
    print('\nStage 2 — Dense retrieval (bge-large + FAISS) ...')
    _bge = SentenceTransformer(BGE_MODEL_ID)
    _prefixed  = [BGE_QUERY_PREFIX + t for t in new_q_texts]
    _new_q_emb = _bge.encode(_prefixed, batch_size=BGE_BATCH,
                              show_progress_bar=True,
                              normalize_embeddings=True).astype(np.float32)
    _index = faiss.IndexFlatIP(bge_corpus_emb.shape[1])
    _index.add(bge_corpus_emb)
    _D, _I = _index.search(_new_q_emb, DENSE_K)
    _cids  = np.array(corpus_ids)
    new_dense = {new_q_ids[i]: _cids[_I[i]].tolist() for i in range(len(new_q_ids))}

    # Stage 3: RRF
    print('\nStage 3 — RRF fusion ...')
    new_rrf = {}
    for qid in new_q_ids:
        rrf_scores: dict[str, float] = {}
        for rank, doc_id in enumerate(new_sparse.get(qid, []), start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_C + rank)
        for rank, doc_id in enumerate(new_dense.get(qid, []), start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_C + rank)
        new_rrf[qid] = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:FUSION_K]

    # Stage 4: cross-encoder
    print('\nStage 4 — Cross-encoder reranking ...')
    _ce = CrossEncoder(CE_MODEL_ID, max_length=512)
    _new_qid_to_text = dict(zip(new_queries['doc_id'],
                                new_queries.apply(format_text, axis=1)))
    new_reranked = {}
    for qid in tqdm(new_q_ids, desc='CE reranking'):
        cands  = new_rrf.get(qid, [])
        if not cands:
            new_reranked[qid] = []; continue
        q_text = _new_qid_to_text.get(qid, '')
        pairs  = [[q_text, _cid_to_text.get(cid, '')] for cid in cands]
        logits = _ce.predict(pairs, batch_size=CE_BATCH, show_progress_bar=False)
        new_reranked[qid] = [cands[i] for i in np.argsort(-logits)[:FINAL_K]]

    with open(output_path, 'w') as _f:
        json.dump(new_reranked, _f)
    print(f'\nFinal submission saved to: {output_path}')
    return new_reranked


# Usage:
# 1. Run bm25.ipynb on the held-out queries to produce the sparse JSON.
# 2. Then call:
#
final_sub = run_pipeline(
    new_queries_path='../data/queries.parquet',
    sparse_path='../submissions/submission_sparse_routed_top100.json',
)

print('run_pipeline() defined and ready.')
print('Provide the sparse JSON from bm25.ipynb via the sparse_path argument.')


Loaded 100 held-out queries.

Stage 1 — Loading sparse routed submission ...
  Loaded 100 queries. ✓

Stage 2 — Dense retrieval (bge-large + FAISS) ...


Batches: 100%|██████████| 4/4 [03:15<00:00, 48.76s/it]



Stage 3 — RRF fusion ...

Stage 4 — Cross-encoder reranking ...


CE reranking:   5%|▌         | 5/100 [02:32<48:10, 30.43s/it]


KeyboardInterrupt: 